## Analyzing Quality of Clustering

TODO:

 - massive convertation of references to PMIDs

Lotte et al., 2007 - no references stored in Pubmed, the same also for Lotte et al., 2018 :(

In [3]:
"""
Closed-loop brain training: the science of neurofeedback (2017)

Two distinct areas of clinical applications are discussed: 
 - ADHD (attention deficit hyperactivity disorder)
 - rehabilitation in stroke.
"""
CLUSTERS = {
    'neurofeedback': {
        'adhd': [7786929, 19712709, 19207632, 23665196, 24399101, 26748531],
        'Stroke': [22401758, 23494615, 24217509]
    }
}

In [4]:
import gzip
import itertools
import json

from collections import Counter

from pysrc.papers.analyzer import KeyPaperAnalyzer
from pysrc.papers.pubtrends_config import PubtrendsConfig, AnalyzerSettings
from pysrc.papers.db.loaders import Loaders
from pysrc.papers.utils import vectorize_corpus, compute_tfidf

In [5]:
PUBTRENDS_CONFIG = PubtrendsConfig(test=False)


def reload_exported_analyzer(path_to_archive, source='Pubmed'):
    """
    Load analysis data from json.gz archive generated by PubTrends.
    """
    with gzip.open(path_to_archive, 'rt', encoding='UTF-8') as zipfile:
        data = json.load(zipfile)

    loader, url_prefix = Loaders.get_loader_and_url_prefix(source, PUBTRENDS_CONFIG)
    analyzer = KeyPaperAnalyzer(loader, PUBTRENDS_CONFIG)
    analyzer.init(data)
    
    # Fill other fields (TODO: move to init)
    analyzer.ids = set(analyzer.df['id'])
    analyzer.n_papers = len(analyzer.ids)
    analyzer.pub_types = list(set(analyzer.df['type']))
    analyzer.query = 'restored from PubTrends export'
    
    PUB_DF_COLUMNS = ['id', 'title', 'abstract', 'year', 'type', 'keywords', 'mesh', 'doi', 'aux']
    analyzer.pub_df = analyzer.df[PUB_DF_COLUMNS]
    
    analyzer.components = set(analyzer.df['comp'].unique())
    if -1 in analyzer.components:
        analyzer.components.remove(-1)
        
    settings = AnalyzerSettings()
    analyzer.corpus_ngrams, analyzer.corpus_counts = \
        vectorize_corpus(analyzer.pub_df,
                         max_features=settings.VECTOR_WORDS, n_gram=settings.VECTOR_NGRAMS,
                         min_df=settings.VECTOR_MIN_DF, max_df=settings.VECTOR_MAX_DF)
    tfidf = compute_tfidf(analyzer.corpus_counts)
    analyzer.texts_similarity = analyzer.analyze_texts_similarity(analyzer.pub_df, tfidf,
                                                                  settings.SIMILARITY_TEXT_MIN,
                                                                  settings.SIMILARITY_TEXT_CITATION_N)
    
    return analyzer

In [6]:
PATH_TO_ZIP = '/home/willenjoy/work/pubtrends/local/pubtrends_export/64.json.gz'
REVIEW_ID = '28003656'

analyzer = reload_exported_analyzer(PATH_TO_ZIP)

In [7]:
len(analyzer.components) # Equals to 11 in the current PubTrends setup

1

## Number of components

In [6]:
param_grid = {
    'SIMILARITY_BIBLIOGRAPHIC_COUPLING': [0.01, 0.1, 1, 10, 100],
    'SIMILARITY_COCITATION': [0.01, 0.1, 1, 10, 100],
    'SIMILARITY_CITATION': [0.01, 0.1, 1, 10, 100],
    'SIMILARITY_TEXT_CITATION': [0.01, 0.1, 1, 10, 100]
}

In [7]:
def settings_of_interest(analyzer_settings):
    return (
        analyzer_settings.SIMILARITY_BIBLIOGRAPHIC_COUPLING,
        analyzer_settings.SIMILARITY_COCITATION,
        analyzer_settings.SIMILARITY_CITATION,
        analyzer_settings.SIMILARITY_TEXT_CITATION,
    )

In [8]:
baseline_params = dict(
    TOPIC_MIN_SIZE=0,
    TOPICS_MAX_NUMBER=500
)

In [ ]:
param_names = param_grid.keys()
result = {}

# Iterate over param grid
for param_values in itertools.product(*param_grid.values()):
    params = {k: v for k, v in zip(param_names, param_values)}
    for k, v in baseline_params.items():
        params[k] = v
    
    settings = AnalyzerSettings(**params)
    analyzer.analyze_papers(analyzer.ids, analyzer.query, 
                            settings=settings, load_data=False, compute_topic_descriptions=False)

    soi = settings_of_interest(settings)
    comps = len(analyzer.components)
    sizes = Counter(analyzer.partition.values())
    max_size = max(sizes.values())
    min_size = min(sizes.values())
    print(f'{soi} - {comps} components, max size = {max_size}, min_size = {min_size}')
    result[soi] = (comps, max_size, min_size)

## Experiments with Clustering

**Ground Truth**

References split into clusters according to their occurrences in text sections.

In [95]:
"""
Closed-loop brain training: the science of neurofeedback (2017)

Clustering based on occurrence in different sections.
Calculated below in the 'Grobid Reference Extraction' section.
"""

print(SECTION_CLUSTERING)

{'20692351': 0, '21617042': 0, '24728268': 0, '23253623': 0, '16838014': 0, '27074513': 0, '23236202': 0, '23022326': 0, '15889581': 0, '12824763': 0, '17626211': 0, '25762907': 0, '26348555': 0, '17336094': 0, '22458676': 0, '21840399': 0, '16791142': 0, '16899397': 0, '17133383': 0, '25870552': 0, '22021045': 0, '20888628': 0, '25519874': 0, '15852014': 0, '22388818': 1, '23954030': 1, '16792290': 1, '17057705': 1, '25796342': 1, '20570245': 1, '24231399': 1, '20384819': 1, '21903976': 1, '23536382': 1, '16791145': 1, '19540746': 1, '25566028': 1, '27620975': 1, '23719815': 1, '26971473': 1, '10404201': 1, '17234689': 1, '24151462': 2, '687682': 2, '7932682': 2, '3567234': 2, '24705203': 2, '26948894': 2, '26500521': 2, '20977537': 2, '23966933': 2, '26066840': 2, '15978025': 3, '25673851': 3, '23583615': 3, '22732558': 3, '25566020': 3, '6487671': 4, '11449024': 4, '10454276': 4, '26748531': 4, '7786929': 4, '19712709': 4, '24399101': 4, '19207632': 4, '22877086': 4, '23665196': 4, 

In [143]:
print(APPLICATION_CLUSTERING)

{'6487671': 0, '11449024': 0, '10454276': 0, '26748531': 0, '7786929': 0, '19712709': 0, '24399101': 0, '19207632': 0, '22877086': 0, '23665196': 0, '19715181': 0, '25220088': 0, '22617866': 0, '24321363': 0, '25870550': 0, '22608481': 0, '25461225': 0, '25329936': 0, '23518223': 0, '15039008': 0, '23264371': 0, '18762860': 0, '26706052': 0, '15827991': 0, '24103255': 0, '23575842': 0, '17919929': 0, '20538065': 0, '17670949': 0, '12738893': 1, '23532011': 1, '25712802': 1, '22645108': 1, '26671217': 1, '22401758': 1, '20303409': 1, '23565083': 1, '26219602': 1, '27071949': 1}


### How to evaluate clustering?

**Rand index** (0 - bad, 1 - good)

Each pair of papers belongs to either same or different cluster. Rand index denotes accuracy of pair classification in terms of same/different cluster compared to ground truth.

**Adjusted Rand index** (-1 - bad, 0 - random, 1 - good)

A version of Rand index adjusted against random clustering.

**TODO: add other metrics and check their correlation: https://scikit-learn.org/stable/modules/clustering.html#clustering-performance-evaluation**

In [144]:
from sklearn.metrics.cluster import adjusted_rand_score, contingency_matrix, pair_confusion_matrix, v_measure_score

ImportError: cannot import name 'pair_confusion_matrix' from 'sklearn.metrics.cluster' (/home/willenjoy/anaconda3/envs/pubtrends/lib/python3.8/site-packages/sklearn/metrics/cluster/__init__.py)

In [98]:
def get_clustering(analyzer, target_ids):
    data = analyzer.df[analyzer.df['id'].isin(target_ids)]
    return {k: v for k, v in zip(data['id'], data['comp'])}

In [120]:
def evaluate_clustering(analyzer, ground_truth):
    actual_clustering = get_clustering(analyzer, ground_truth.keys())
    
    # Align clusterings
    labels_true = []
    labels_pred = []

    for pmid in actual_clustering:
        labels_true.append(ground_truth[pmid])
        labels_pred.append(actual_clustering[pmid])
        
    # Evaluate (Rand index, contingency matrix)
    rand_score = adjusted_rand_score(labels_true, labels_pred)
    cm = contingency_matrix(labels_true, labels_pred) 
    
    return score, cm

## Current PubTrends Results

In [92]:
analyzer = reload_exported_analyzer(PATH_TO_ZIP)

In [121]:
evaluate_clustering(analyzer, SECTION_CLUSTERING)

(0.08662828001699899,
 array([[ 8,  0,  0, 10,  0,  3,  3],
        [ 8,  0,  0,  5,  2,  0,  3],
        [ 6,  0,  0,  4,  0,  0,  0],
        [ 4,  0,  0,  1,  0,  0,  0],
        [26,  2,  1,  0,  0,  0,  0],
        [ 7,  0,  0,  3,  0,  0,  0]]))

In [122]:
evaluate_clustering(analyzer, APPLICATION_CLUSTERING)

(0.15625716688326594,
 array([[26,  2,  1,  0],
        [ 7,  0,  0,  3]]))

## Grid Search

In [123]:
param_grid = {
    'SIMILARITY_BIBLIOGRAPHIC_COUPLING': [0.01, 0.1, 1, 10, 100],
    'SIMILARITY_COCITATION': [0.01, 0.1, 1, 10, 100],
    'SIMILARITY_CITATION': [0.01, 0.1, 1, 10, 100],
    'SIMILARITY_TEXT_CITATION': [0.01, 0.1, 1, 10, 100]
}

In [124]:
def settings_of_interest(analyzer_settings):
    return (
        analyzer_settings.SIMILARITY_BIBLIOGRAPHIC_COUPLING,
        analyzer_settings.SIMILARITY_COCITATION,
        analyzer_settings.SIMILARITY_CITATION,
        analyzer_settings.SIMILARITY_TEXT_CITATION,
    )

In [125]:
baseline_params = dict(
    TOPIC_MIN_SIZE=0,
    TOPICS_MAX_NUMBER=500
)

In [126]:
param_names = param_grid.keys()
best_score = 0
best_cm = None

# Iterate over param grid
for param_values in itertools.product(*param_grid.values()):
    params = {k: v for k, v in zip(param_names, param_values)}
    for k, v in baseline_params.items():
        params[k] = v
    
    settings = AnalyzerSettings(**params)
    analyzer.analyze_papers(analyzer.ids, analyzer.query, 
                            settings=settings, load_data=False, compute_topic_descriptions=False)

    soi = settings_of_interest(settings)
    score, cm = evaluate_clustering(analyzer, SECTION_CLUSTERING)
    new_best = False
    if score > best_score:
        best_score = score
        best_cm = cm
        new_best = True
        
    print(f"{'BEST' if new_best else ''} {soi} - score {score}")

BEST (0.01, 0.01, 0.01, 0.01) - score 0.0801334912612967
 (0.01, 0.01, 0.01, 0.1) - score -0.016303851078282874
 (0.01, 0.01, 0.01, 1) - score -0.017370943189474006
 (0.01, 0.01, 0.01, 10) - score -0.00982446994008097
 (0.01, 0.01, 0.01, 100) - score -0.00468824464326378
 (0.01, 0.01, 0.1, 0.01) - score -0.011328759996229298
 (0.01, 0.01, 0.1, 0.1) - score -0.016303851078282874
 (0.01, 0.01, 0.1, 1) - score -0.017370943189474006
 (0.01, 0.01, 0.1, 10) - score 0.002084368405978355
 (0.01, 0.01, 0.1, 100) - score -0.00982446994008097
 (0.01, 0.01, 1, 0.01) - score 0.06413878652430291
BEST (0.01, 0.01, 1, 0.1) - score 0.08589235482960189
BEST (0.01, 0.01, 1, 1) - score 0.10851901813546126
BEST (0.01, 0.01, 1, 10) - score 0.1390230976392479
 (0.01, 0.01, 1, 100) - score -0.005632979386033617
 (0.01, 0.01, 10, 0.01) - score 0.11597941347213914
BEST (0.01, 0.01, 10, 0.1) - score 0.16339635305962635
 (0.01, 0.01, 10, 1) - score 0.06968637303413397
BEST (0.01, 0.01, 10, 10) - score 0.203296233

 (0.1, 0.1, 1, 0.1) - score -0.011328759996229298
 (0.1, 0.1, 1, 1) - score -0.016303851078282874
 (0.1, 0.1, 1, 10) - score -0.017370943189474006
 (0.1, 0.1, 1, 100) - score 0.002084368405978355
 (0.1, 0.1, 10, 0.01) - score 0.06413878652430291
 (0.1, 0.1, 10, 0.1) - score 0.06413878652430291
 (0.1, 0.1, 10, 1) - score 0.08589235482960189
 (0.1, 0.1, 10, 10) - score 0.10851901813546126
 (0.1, 0.1, 10, 100) - score 0.1390230976392479
 (0.1, 0.1, 100, 0.01) - score 0.11597941347213914
 (0.1, 0.1, 100, 0.1) - score 0.11597941347213914
 (0.1, 0.1, 100, 1) - score 0.16339635305962635
 (0.1, 0.1, 100, 10) - score 0.06968637303413397
 (0.1, 0.1, 100, 100) - score 0.20329623384707662
 (0.1, 1, 0.01, 0.01) - score -0.013229290951640814
 (0.1, 1, 0.01, 0.1) - score -0.013229290951640814
 (0.1, 1, 0.01, 1) - score -0.013229290951640814
 (0.1, 1, 0.01, 10) - score -0.013229290951640814
 (0.1, 1, 0.01, 100) - score 0.0835234186482975
 (0.1, 1, 0.1, 0.01) - score -0.013229290951640814
 (0.1, 1, 0.1

 (1, 10, 0.1, 0.01) - score -0.013229290951640814
 (1, 10, 0.1, 0.1) - score -0.013229290951640814
 (1, 10, 0.1, 1) - score -0.013229290951640814
 (1, 10, 0.1, 10) - score -0.013229290951640814
 (1, 10, 0.1, 100) - score -0.013229290951640814
 (1, 10, 1, 0.01) - score -0.013229290951640814
 (1, 10, 1, 0.1) - score -0.013229290951640814
 (1, 10, 1, 1) - score -0.013229290951640814
 (1, 10, 1, 10) - score -0.013229290951640814
 (1, 10, 1, 100) - score -0.013229290951640814
 (1, 10, 10, 0.01) - score -0.013229290951640814
 (1, 10, 10, 0.1) - score -0.013229290951640814
 (1, 10, 10, 1) - score -0.013229290951640814
 (1, 10, 10, 10) - score -0.013229290951640814
 (1, 10, 10, 100) - score -0.013745276123673496
 (1, 10, 100, 0.01) - score -0.013229290951640814
 (1, 10, 100, 0.1) - score -0.013229290951640814
 (1, 10, 100, 1) - score -0.013229290951640814
 (1, 10, 100, 10) - score -0.013229290951640814
 (1, 10, 100, 100) - score -0.013229290951640814
 (1, 100, 0.01, 0.01) - score -0.0137452761

 (100, 0.01, 0.01, 0.01) - score 0.11329134902220751
 (100, 0.01, 0.01, 0.1) - score 0.10716398440699562
 (100, 0.01, 0.01, 1) - score 0.0660966651185739
 (100, 0.01, 0.01, 10) - score 0.16954559630855717
 (100, 0.01, 0.01, 100) - score 0.0747897057193875
 (100, 0.01, 0.1, 0.01) - score -0.013800171855103367
 (100, 0.01, 0.1, 0.1) - score 0.0149476308645859
 (100, 0.01, 0.1, 1) - score 0.04158413503405076
 (100, 0.01, 0.1, 10) - score 0.07075025767582023
 (100, 0.01, 0.1, 100) - score 0.07559474183122239
 (100, 0.01, 1, 0.01) - score 0.012583169145741936
 (100, 0.01, 1, 0.1) - score 0.012583169145741936
 (100, 0.01, 1, 1) - score 0.06741189866540277
 (100, 0.01, 1, 10) - score 0.11544749366813069
 (100, 0.01, 1, 100) - score 0.0922387603416321
 (100, 0.01, 10, 0.01) - score 0.08609571002289726
 (100, 0.01, 10, 0.1) - score 0.08609571002289726
 (100, 0.01, 10, 1) - score 0.08609571002289726
 (100, 0.01, 10, 10) - score 0.08336305903371659
 (100, 0.01, 10, 100) - score 0.1299887557327732

In [132]:
# Adjusted Rand index > 0.2

BEST_SOI = (0.01, 0.1, 100, 100)
BEST_SCORE = 0.2153548037588293
SOI_GOOD = [
    (0.01, 0.01, 10, 10),
    (0.1, 0.01, 10, 10),
    (0.1, 0.01, 100, 100),
    (0.1, 0.1, 100, 100),
    (1, 0.1, 100, 100)
]

In [133]:
params = {k: v for k, v in zip(param_names, BEST_SOI)}
for k, v in baseline_params.items():
    params[k] = v

settings = AnalyzerSettings(**params)
analyzer.analyze_papers(analyzer.ids, analyzer.query, 
                        settings=settings, load_data=False, compute_topic_descriptions=False)

soi = settings_of_interest(settings)
score, cm = evaluate_clustering(analyzer, SECTION_CLUSTERING)

print(score)
print(cm)

0.2153548037588293
[[ 0  3  0  1  9  4  0  2  0  0  2  0  0  3  0  0  0  0]
 [ 0  2  0  0  3  5  2  3  0  0  0  1  0  0  2  0  0  0]
 [ 0  0  1  0  3  0  0  0  0  0  0  0  2  0  0  2  1  1]
 [ 0  0  0  2  1  0  0  0  0  0  0  0  1  0  1  0  0  0]
 [ 2  7  0 17  0  0  0  2  1  0  0  0  0  0  0  0  0  0]
 [ 0  6  0  0  1  0  0  0  0  1  0  0  2  0  0  0  0  0]]


APPLICATION_CLUSTERING

In [134]:
param_names = param_grid.keys()
best_soi = None
best_score = 0
best_cm = None

# Iterate over param grid
for param_values in itertools.product(*param_grid.values()):
    params = {k: v for k, v in zip(param_names, param_values)}
    for k, v in baseline_params.items():
        params[k] = v
    
    settings = AnalyzerSettings(**params)
    analyzer.analyze_papers(analyzer.ids, analyzer.query, 
                            settings=settings, load_data=False, compute_topic_descriptions=False)

    soi = settings_of_interest(settings)
    score, cm = evaluate_clustering(analyzer, APPLICATION_CLUSTERING)
    new_best = False
    if score > best_score:
        best_soi = soi
        best_score = score
        best_cm = cm
        new_best = True
        
    print(f"{'BEST' if new_best else ''} {soi} - score {score}")

BEST (0.01, 0.01, 0.01, 0.01) - score 0.07994211145190078
 (0.01, 0.01, 0.01, 0.1) - score -0.07826828197246387
 (0.01, 0.01, 0.01, 1) - score -0.08314703353396397
 (0.01, 0.01, 0.01, 10) - score 0.01530604076274037
 (0.01, 0.01, 0.01, 100) - score 0.016644018496182716
 (0.01, 0.01, 0.1, 0.01) - score -0.05855275921059594
 (0.01, 0.01, 0.1, 0.1) - score -0.07826828197246387
 (0.01, 0.01, 0.1, 1) - score -0.08314703353396397
 (0.01, 0.01, 0.1, 10) - score 0.041607400311571674
 (0.01, 0.01, 0.1, 100) - score 0.01530604076274037
 (0.01, 0.01, 1, 0.01) - score 0.06747612922117988
 (0.01, 0.01, 1, 0.1) - score 0.07994211145190078
BEST (0.01, 0.01, 1, 1) - score 0.14820943551695956
BEST (0.01, 0.01, 1, 10) - score 0.21081578348649266
 (0.01, 0.01, 1, 100) - score 0.01530604076274037
 (0.01, 0.01, 10, 0.01) - score 0.09223586607375399
 (0.01, 0.01, 10, 0.1) - score 0.1721104416289059
 (0.01, 0.01, 10, 1) - score 0.015446714188559493
BEST (0.01, 0.01, 10, 10) - score 0.22459928577360685
 (0.01

 (0.1, 0.1, 1, 10) - score -0.08314703353396397
 (0.1, 0.1, 1, 100) - score 0.041607400311571674
 (0.1, 0.1, 10, 0.01) - score 0.06747612922117988
 (0.1, 0.1, 10, 0.1) - score 0.06747612922117988
 (0.1, 0.1, 10, 1) - score 0.07994211145190078
 (0.1, 0.1, 10, 10) - score 0.14820943551695956
 (0.1, 0.1, 10, 100) - score 0.21081578348649266
 (0.1, 0.1, 100, 0.01) - score 0.09223586607375399
 (0.1, 0.1, 100, 0.1) - score 0.09223586607375399
 (0.1, 0.1, 100, 1) - score 0.1721104416289059
 (0.1, 0.1, 100, 10) - score 0.015446714188559493
 (0.1, 0.1, 100, 100) - score 0.22459928577360685
 (0.1, 1, 0.01, 0.01) - score -0.05855275921059594
 (0.1, 1, 0.01, 0.1) - score -0.05855275921059594
 (0.1, 1, 0.01, 1) - score -0.05855275921059594
 (0.1, 1, 0.01, 10) - score -0.05855275921059594
 (0.1, 1, 0.01, 100) - score 0.1513937499246567
 (0.1, 1, 0.1, 0.01) - score -0.05855275921059594
 (0.1, 1, 0.1, 0.1) - score -0.05855275921059594
 (0.1, 1, 0.1, 1) - score -0.05855275921059594
 (0.1, 1, 0.1, 10) -

 (1, 10, 0.1, 10) - score -0.05855275921059594
 (1, 10, 0.1, 100) - score -0.05855275921059594
 (1, 10, 1, 0.01) - score -0.05855275921059594
 (1, 10, 1, 0.1) - score -0.05855275921059594
 (1, 10, 1, 1) - score -0.05855275921059594
 (1, 10, 1, 10) - score -0.05855275921059594
 (1, 10, 1, 100) - score -0.05855275921059594
 (1, 10, 10, 0.01) - score -0.05855275921059594
 (1, 10, 10, 0.1) - score -0.05855275921059594
 (1, 10, 10, 1) - score -0.05855275921059594
 (1, 10, 10, 10) - score -0.05855275921059594
 (1, 10, 10, 100) - score -0.06106668429052931
 (1, 10, 100, 0.01) - score -0.05855275921059594
 (1, 10, 100, 0.1) - score -0.05855275921059594
 (1, 10, 100, 1) - score -0.05855275921059594
 (1, 10, 100, 10) - score -0.05855275921059594
 (1, 10, 100, 100) - score -0.05855275921059594
 (1, 100, 0.01, 0.01) - score -0.06106668429052931
 (1, 100, 0.01, 0.1) - score -0.06106668429052931
 (1, 100, 0.01, 1) - score -0.06106668429052931
 (1, 100, 0.01, 10) - score -0.06106668429052931
 (1, 100

 (100, 0.01, 0.01, 100) - score 0.13520597002981008
 (100, 0.01, 0.1, 0.01) - score -0.14665053036235495
 (100, 0.01, 0.1, 0.1) - score -0.07651584171998835
 (100, 0.01, 0.1, 1) - score 0.05522679712294026
 (100, 0.01, 0.1, 10) - score 0.13067637478444144
 (100, 0.01, 0.1, 100) - score 0.13520597002981008
 (100, 0.01, 1, 0.01) - score -0.06733696153491373
 (100, 0.01, 1, 0.1) - score -0.06733696153491373
 (100, 0.01, 1, 1) - score 0.10037667193569255
 (100, 0.01, 1, 10) - score 0.10733615911313994
 (100, 0.01, 1, 100) - score 0.10163819447923456
 (100, 0.01, 10, 0.01) - score 0.03077082362685713
 (100, 0.01, 10, 0.1) - score 0.03077082362685713
 (100, 0.01, 10, 1) - score 0.03077082362685713
 (100, 0.01, 10, 10) - score 0.14537594010925645
 (100, 0.01, 10, 100) - score 0.24633487324020836
 (100, 0.01, 100, 0.01) - score 0.2060695507794484
 (100, 0.01, 100, 0.1) - score 0.2060695507794484
 (100, 0.01, 100, 1) - score 0.2060695507794484
 (100, 0.01, 100, 10) - score 0.2060695507794484
 (

In [136]:
print(best_score)
print(best_soi)
print(best_cm)

0.40152849381451006
(100, 0.1, 0.01, 100)
[[ 1  2 23  1  0  2]
 [ 7  0  2  0  1  0]]


In [142]:
params = {k: v for k, v in zip(param_names, best_soi)}
for k, v in baseline_params.items():
    params[k] = v

settings = AnalyzerSettings(**params)
analyzer.analyze_papers(analyzer.ids, analyzer.query, 
                        settings=settings, load_data=False, compute_topic_descriptions=False)

print(analyzer.components)

{0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30}


## OLD - Experiments with Manual Clustering

In [11]:
marked_refs = []
for refs in CLUSTERS.values():
    marked_refs.extend([str(r) for r in refs])

In [12]:
marked_refs

['adhd', 'Stroke']

In [7]:
for _, row in analyzer.df_kwd.iterrows():
    keywords = list(map(lambda el: el[0], row['kwd']))
    print(row['comp'] + 1, ",".join(keywords))

1 adhd,eeg,bci,neurofeedback,child,training,hyperactivity,attention,treatment,motor,rhythm,alpha,group,control,activity,interface,study,theta,performance,smr
2 network,state,functional,connectivity,task,method,data,signal,analysis,region,connection,fmri,global,magnetic,resonance,fcmri,correlation,image,default,mode
3 network,task,insula,anterior,region,emotion,control,function,attention,activity,activation,study,cingulate,neural,prefrontal,analysis,cognitive,adult,default,magnetic
4 reward,dopamine,neuron,striatum,decision,activity,animal,value,memory,cue,nucleus,behavior,task,area,outcome,risk,neural,tegmental,basal,magnetic
5 gamma,activity,neuron,spike,eeg,power,potential,signal,visual,theta,cell,phase,bold,method,frequency,memory,band,lfp,neural,fmri
6 neurofeedback,rtfmri,fmri,activity,time,feedback,training,real,emotion,magnetic,motor,resonance,method,imagery,connectivity,pain,functional,participant,regulation,region
7 sleep,effect,neuron,synaptic,metabolism,change,drug,motor,act

In [8]:
analyzer.df[analyzer.df['id'].isin(marked_refs)][['id', 'title', 'comp']]

NameError: name 'marked_refs' is not defined

In [105]:
PATH_TO_ZIP_ZOOMED = '/home/willenjoy/work/pubtrends/local/pubtrends_export/Neurofeedback-Ros-2017_subtopic1_detailed.zip'

zoomed_analyzer = reload_exported_analyzer(PATH_TO_ZIP_ZOOMED)

In [108]:
for _, row in zoomed_analyzer.df_kwd.iterrows():
    keywords = list(map(lambda el: el[0], row['kwd']))
    print(row['comp'] + 1, ",".join(keywords))

1 adhd,child,attention,eeg,training,hyperactivity,disorder,theta,effect,deficit,group,beta,treatment,task,adult,alpha,cortical,parent,performance,active
2 pain,fmri,time,theta,active,regulation,motor,effect,real,adult,feedback,training,participant,imagery,magnetic,group,neural,self,resonance,functional
3 rtfmri,emotion,fmri,time,magnetic,real,resonance,training,feedback,regulation,functional,region,anterior,insula,self,adult,motor,participant,activation,computer
4 motor,bci,hand,movement,neuron,predictor,computer,skill,corticostriatal,interface,performance,function,neuroprosthetic,imagery,group,rhythm,innervation,coherence,extremity,adult


In [106]:
zoomed_analyzer.df[zoomed_analyzer.df['id'].isin(marked_refs)][['id', 'title', 'comp']]

,id,title,comp
1,19207632,Is neurofeedback an efficacious treatment for ...,0
29,7786929,Evaluation of the effectiveness of EEG neurofe...,0
39,24217509,Brain repair after stroke--a novel neurologica...,0
60,23494615,Brain-machine interface in chronic stroke reha...,3
61,24399101,Neurofeedback and cognitive attention training...,0
69,23665196,Neurofeedback and standard pharmacological int...,0
72,26748531,A randomized controlled trial into the effects...,0
90,22401758,Investigation of fMRI neurofeedback of differe...,2
104,19712709,Distinct EEG effects related to neurofeedback ...,0


In [109]:
NEUROFEEDBACK_CLUSTERS

{'ADHD': [7786929, 19712709, 19207632, 23665196, 24399101, 26748531],
 'Stroke': [22401758, 23494615, 24217509]}

## Grobid Reference Extraction

In [6]:
import requests

from bs4 import BeautifulSoup

In [7]:
GROBID_API_URL = 'http://localhost:8070/api/processReferences'
PDF_FILE = '/home/willenjoy/Downloads/nrn.2016.164.pdf'
XML_FILE = '/home/willenjoy/Downloads/nrn.2016.164-refs.tei.xml'

In [8]:
"""
TODO: retry

https://github.com/kermitt2/grobid_client_python/blob/master/grobid_client.py
"""

def extract_references(pdf_file):
    files = {
        'input': {
            pdf_file,
            open(pdf_file, 'rb'),
            'application/pdf',
            {'Expires': '0'}
        }
    }
    
    res, status = requests.post(
        GROBID_API_URL,
        files=files,
        headers={'Accept': 'application/xml'}
    )
    
    return res, status

In [9]:
# res, status = extract_references(PDF_FILE)

In [10]:
with open(XML_FILE, 'r') as refs_xml:
    soup = BeautifulSoup(refs_xml, 'xml')

In [62]:
refs = soup.select('biblStruct')

In [63]:
len(refs)

222

In [34]:
pubmed_ref_ids = list(analyzer.citations_graph.successors(REVIEW_ID))

In [49]:
pubmed_data = analyzer.df[analyzer.df['id'].isin(pubmed_ref_ids)]
pubmed_ref_search_index = {v.lower(): k for k, v in zip(pubmed_data['id'], pubmed_data['title'])}
pubmed_ref_titles = {k: v for k, v in zip(pubmed_data['id'], pubmed_data['title'])}

In [52]:
pubmed_ref_titles['20692351'].lower() in pubmed_ref_search_index

True

In [73]:
found = 0
num2pmid = {}

for i, ref in enumerate(refs):
    if ref.analytic and ref.analytic.title:
        ref_title = ref.analytic.title.text
        if ref_title.lower() in pubmed_refs:
            pmid = pubmed_refs[ref_title.lower()]
            print(i + 1, ref_title, pmid)
            num2pmid[i + 1] = pmid
            found += 1
        
print(f'Found {found} references')

4 Real-time support vector classification and feedback of multiple emotional brain states 20692351
5 Online decoding of object-based attention using real-time fMRI 24438492
7 Neurofeedback as a nonpharmacological treatment for adults with attention-deficit/hyperactivity disorder (ADHD): study protocol for a randomized controlled trial 25928870
10 Real-time fMRI neurofeedback: progress and challenges 23541800
11 Selective attention from voluntary control of neurons in prefrontal cortex 21617042
12 Volitional modulation of optically recorded calcium signals during neuroprosthetic learning 24728268
13 High-performance neuroprosthetic control by an individual with tetraplegia 23253623
14 Neuronal ensemble control of prosthetic devices by a human with tetraplegia 16838014
15 Restoring cortical control of functional movement in a human with quadriplegia 27074513
16 Effects of neural synchrony on surface EEG 23236202
18 Mind over chatter: plastic up-regulation of the fMRI salience network dir

In [74]:
num2pmid

{4: '20692351',
 5: '24438492',
 7: '25928870',
 10: '23541800',
 11: '21617042',
 12: '24728268',
 13: '23253623',
 14: '16838014',
 15: '27074513',
 16: '23236202',
 18: '23022326',
 19: '15889581',
 20: '12824763',
 21: '17626211',
 22: '25762907',
 23: '26348555',
 24: '17336094',
 25: '22458676',
 28: '21840399',
 30: '16791142',
 31: '16899397',
 32: '17133383',
 33: '25870552',
 34: '22021045',
 35: '20888628',
 39: '25519874',
 40: '15852014',
 45: '22388818',
 46: '23954030',
 47: '16792290',
 48: '17057705',
 51: '25796342',
 53: '20570245',
 54: '24231399',
 55: '20384819',
 56: '21903976',
 58: '23536382',
 59: '16791145',
 60: '19540746',
 61: '25566028',
 62: '27620975',
 63: '23719815',
 65: '26971473',
 66: '10404201',
 67: '17234689',
 69: '21964375',
 71: '24151462',
 74: '687682',
 75: '7932682',
 76: '3567234',
 79: '24705203',
 80: '26948894',
 81: '26500521',
 82: '20977537',
 84: '23966933',
 85: '26066840',
 86: '15978025',
 90: '25673851',
 91: '23583615',
 92:

In [85]:
"""
FIXME:
 - how to resolve multiple occurrences in text? (currently use first occurrence)
 - numbers shift in current enumerate implementation, use ref ids in XML (after 78 +1, after 120 +3)
"""

SECTION_CLUSTERING_RAW = {
    'Neural specificity and plasticity': {
        'The neural substrates of self-regulation': [3, 4, 6] + list(range(11, 44)),
        'Neural plasticity and specificity': list(range(44, 68)),
    },
    'Neural mechanisms of self-regulation': {
        'Factors influencing neurofeedback learning': list(range(70, 86)),
        'Neural substrates of self-regulation': list(range(86, 94)),
    },
    'Clinical applications of neurofeedback': {
        'Attention deficit hyperactivity disorder': list(range(94, 137)),
        'Rehabilitation in stroke': list(range(138, 156)),
    }
}

In [86]:
clusters = []

for section_refs in SECTION_CLUSTERING_RAW.values():
    for subsection, raw_refs in section_refs.items():
        cluster = []
        for ref in raw_refs:
            if ref in num2pmid:
                cluster.append(num2pmid[ref])
        clusters.append(cluster)

In [87]:
for i, cluster in enumerate(clusters):
    print(f'Cluster {i + 1}')
    print(analyzer.df[analyzer.df['id'].isin(cluster)][['id', 'title']])

Cluster 1
           id                                              title
36   23253623  High-performance neuroprosthetic control by an...
64   20888628  Reorganization of functional and effective con...
73   22458676  Volitional reduction of anterior cingulate cor...
77   23236202         Effects of neural synchrony on surface EEG
86   15889581  Increasing individual upper alpha power by neu...
144  17133383    Real-time fMRI using brain-state classification
168  12824763  Ecological validity of neurofeedback: modulati...
174  25762907  Improvement in precision grip force control wi...
221  27074513  Restoring cortical control of functional movem...
233  17626211  Direct instrumental conditioning of neural act...
279  16791142  Decoding mental states from brain activity in ...
302  26348555  Enhanced control of dorsolateral prefrontal co...
303  16838014  Neuronal ensemble control of prosthetic device...
380  16899397  Beyond mind-reading: multi-voxel pattern analy...
385  15852014  

In [114]:
SECTION_CLUSTERING = {}

for i, cluster in enumerate(clusters):
    for ref in cluster:
        SECTION_CLUSTERING[ref] = i

In [115]:
SECTION_CLUSTERING

{'20692351': 0,
 '21617042': 0,
 '24728268': 0,
 '23253623': 0,
 '16838014': 0,
 '27074513': 0,
 '23236202': 0,
 '23022326': 0,
 '15889581': 0,
 '12824763': 0,
 '17626211': 0,
 '25762907': 0,
 '26348555': 0,
 '17336094': 0,
 '22458676': 0,
 '21840399': 0,
 '16791142': 0,
 '16899397': 0,
 '17133383': 0,
 '25870552': 0,
 '22021045': 0,
 '20888628': 0,
 '25519874': 0,
 '15852014': 0,
 '22388818': 1,
 '23954030': 1,
 '16792290': 1,
 '17057705': 1,
 '25796342': 1,
 '20570245': 1,
 '24231399': 1,
 '20384819': 1,
 '21903976': 1,
 '23536382': 1,
 '16791145': 1,
 '19540746': 1,
 '25566028': 1,
 '27620975': 1,
 '23719815': 1,
 '26971473': 1,
 '10404201': 1,
 '17234689': 1,
 '24151462': 2,
 '687682': 2,
 '7932682': 2,
 '3567234': 2,
 '24705203': 2,
 '26948894': 2,
 '26500521': 2,
 '20977537': 2,
 '23966933': 2,
 '26066840': 2,
 '15978025': 3,
 '25673851': 3,
 '23583615': 3,
 '22732558': 3,
 '25566020': 3,
 '6487671': 4,
 '11449024': 4,
 '10454276': 4,
 '26748531': 4,
 '7786929': 4,
 '19712709': 4

In [116]:
APPLICATION_CLUSTERING = {}

for ref in clusters[4]:
    APPLICATION_CLUSTERING[ref] = 0
for ref in clusters[5]:
    APPLICATION_CLUSTERING[ref] = 1

In [117]:
APPLICATION_CLUSTERING

{'6487671': 0,
 '11449024': 0,
 '10454276': 0,
 '26748531': 0,
 '7786929': 0,
 '19712709': 0,
 '24399101': 0,
 '19207632': 0,
 '22877086': 0,
 '23665196': 0,
 '19715181': 0,
 '25220088': 0,
 '22617866': 0,
 '24321363': 0,
 '25870550': 0,
 '22608481': 0,
 '25461225': 0,
 '25329936': 0,
 '23518223': 0,
 '15039008': 0,
 '23264371': 0,
 '18762860': 0,
 '26706052': 0,
 '15827991': 0,
 '24103255': 0,
 '23575842': 0,
 '17919929': 0,
 '20538065': 0,
 '17670949': 0,
 '12738893': 1,
 '23532011': 1,
 '25712802': 1,
 '22645108': 1,
 '26671217': 1,
 '22401758': 1,
 '20303409': 1,
 '23565083': 1,
 '26219602': 1,
 '27071949': 1}

In [8]:
analyzer.similarity_graph.edges(data=True)

EdgeDataView([])